## 1. Locate source code and the Bank Marketing stream


In [ ]:
from pathlib import Path
import os, shutil, zipfile, json, sys, subprocess, time

INPUT_ROOT = Path(os.environ.get("JCAM_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("JCAM_WORK_ROOT", "/kaggle/working"))
CODE = WORK_ROOT / "JCAM" / "Code"
RESULTS = CODE / "results" / "exp10_bank"
BOOTSTRAP_ERROR = None

def extract_matching_zip(target_name, destination):
    matches = []
    for path in INPUT_ROOT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(path) as archive:
                if any(name.replace("\\", "/").endswith(target_name) for name in archive.namelist()):
                    matches.append(path)
        except zipfile.BadZipFile:
            continue
    if len(matches) != 1:
        raise RuntimeError(f"Expected one archive containing {target_name}, found {matches}")
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(matches[0]) as archive:
        for member in archive.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination.resolve()):
                raise RuntimeError(f"Unsafe archive member: {member.filename}")
        archive.extractall(destination)

try:
    source_roots = sorted({path.parent.parent for path in INPUT_ROOT.rglob("run_application.py")
                           if path.parent.name == "kaggle" and path.parent.parent.name == "Code"})
    if not source_roots:
        extracted_source = WORK_ROOT / "_jcam_source"
        extract_matching_zip("Code/kaggle/run_application.py", extracted_source)
        source_roots = sorted({path.parent.parent for path in extracted_source.rglob("run_application.py")
                               if path.parent.name == "kaggle" and path.parent.parent.name == "Code"})
    if len(source_roots) != 1:
        raise RuntimeError(f"Expected one Code source, found {source_roots}")
    shutil.copytree(source_roots[0], CODE, dirs_exist_ok=True)
    inputs = sorted(INPUT_ROOT.rglob("bank_stream.npz"))
    if not inputs:
        extracted_data = WORK_ROOT / "_jcam_bank_input"
        extract_matching_zip("bank_stream.npz", extracted_data)
        inputs = sorted(extracted_data.rglob("bank_stream.npz"))
    if len(inputs) != 1:
        raise RuntimeError(f"Expected one bank_stream.npz, found {inputs}")
    DATA = inputs[0]
    SOURCE_META = DATA.with_suffix(".metadata.json")
    if not SOURCE_META.is_file():
        raise FileNotFoundError(SOURCE_META)
    RESULTS.mkdir(parents=True, exist_ok=True)
    print("Code:", CODE, "Data:", DATA)
except Exception as error:
    BOOTSTRAP_ERROR = repr(error)
    RESULTS.mkdir(parents=True, exist_ok=True)
    print("Bootstrap failed:", BOOTSTRAP_ERROR)


## 2. Validate the chronological NPZ


In [ ]:
import hashlib
import numpy as np

RUN_OK = BOOTSTRAP_ERROR is None
STAGES = []
ROUND_PATH = None

def record(stage, outcome, detail):
    STAGES.append({"stage": stage, "outcome": outcome, "detail": detail})
    print(stage, outcome, detail)

if RUN_OK:
    try:
        source = json.loads(SOURCE_META.read_text(encoding="utf-8"))
        if hashlib.sha256(DATA.read_bytes()).hexdigest() != source["npz_sha256"]:
            raise ValueError("NPZ SHA-256 mismatch")
        with np.load(DATA, allow_pickle=False) as data:
            x, y, g, r, w = [data[k] for k in ("features", "labels", "groups", "radii", "weights")]
            if x.ndim != 3 or y.shape != x.shape[:2] or g.shape != y.shape:
                raise ValueError("Invalid stream shapes")
            if r.shape != (len(x), 3) or w.shape != r.shape:
                raise ValueError("Invalid objective metadata shapes")
            if not np.all(np.isfinite(x)) or not np.all(np.isin(y, [-1, 1])):
                raise ValueError("Invalid features or labels")
            if not np.all((g == 0).any(axis=1) & (g == 1).any(axis=1)):
                raise ValueError("Both age groups are required in every batch")
            if not np.allclose(w.sum(axis=1), 1) or np.any(w < 0) or np.any(r < 0):
                raise ValueError("Invalid weights or radii")
        record("validate", "passed", f"T={len(x)}, n={x.shape[1]}, d={x.shape[2]}")
    except Exception as error:
        RUN_OK = False
        record("validate", "failed", repr(error))
else:
    record("validate", "skipped", BOOTSTRAP_ERROR)


## 3. Run EXP-10 in chronological order


In [ ]:
if RUN_OK:
    started = time.perf_counter()
    command = [sys.executable, str(CODE / "kaggle" / "run_application.py"),
               "--data", str(DATA), "--eta", "0.05", "--output-root", str(RESULTS)]
    outcome = subprocess.run(command, cwd=CODE, text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(outcome.stdout[-5000:])
    if outcome.returncode == 0:
        checkpoints = sorted((RESULTS / "application").rglob("metadata.json"))
        if len(checkpoints) != 1:
            RUN_OK = False
            record("application", "failed", f"Expected one checkpoint, found {checkpoints}")
        else:
            ROUND_PATH = checkpoints[0].with_name("rounds.csv")
            record("application", "passed", f"seconds={time.perf_counter()-started:.2f}")
    else:
        RUN_OK = False
        record("application", "failed", f"exit={outcome.returncode}")
else:
    record("application", "skipped", "earlier gate failed")


## 4. Compute conditional population-band widths


In [ ]:
import base64
CERTIFIER_SOURCE = base64.b64decode("".join([
    'IiIiQ29tcHV0ZSBFWFAtMTAgdW5pZm9ybSBwb3B1bGF0aW9uIGJhbmRzIHVuZGVyIGNvbmRpdGlvbmFsIHdpdGhpbi1iYXRj',
    'aCBJSUQuCgpUaGUgVUNJIGNocm9ub2xvZ3kgZG9lcyBub3QgdmVyaWZ5IHRoaXMgYXNzdW1wdGlvbi4gVGhlIHNjcmlwdCBt',
    'ZWFzdXJlcyBiYW5kCndpZHRoLCBub3QgcG9wdWxhdGlvbiBjb3ZlcmFnZSwgYmVjYXVzZSB0aGUgcG9wdWxhdGlvbiB0YXJn',
    'ZXRzIGFyZSB1bmtub3duLgpTZWUgTWF0aC9zdHJlYW1pbmdfcG9wdWxhdGlvbl9pbnRlcnZhbHMubWQgZm9yIHRoZSBjb25k',
    'aXRpb25hbCBwcm9vZi4KIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNzdgppbXBvcnQganNvbgppbXBvcnQgbWF0aApm',
    'cm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAoKCkZFQVRVUkVTID0gNwpTQ09SRV9CT1VORCA9',
    'IDEyLjAKR0FQX0JPVU5EID0gMTQuMAoKCmRlZiBsb3NzX3JhZGl1cyhuOiBpbnQsIGhvcml6b246IGludCwgZGVsdGFfbG9z',
    'czogZmxvYXQpIC0+IHR1cGxlW2Zsb2F0LCBmbG9hdF06CiAgICBpZiBuIDwgMSBvciBob3Jpem9uIDwgMSBvciBub3QgMCA8',
    'IGRlbHRhX2xvc3MgPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImludmFsaWQgY2VydGlmaWNhdGUgcGFyYW1ldGVy',
    'cyIpCiAgICAjIFNlbGVjdGlvbiBpcyBkZXRlcm1pbmlzdGljIGZyb20gcHVibGljIHNhbXBsZSBzaXplIGFuZCBjb25maWRl',
    'bmNlLgogICAgc3BhY2luZ3MgPSBucC5nZW9tc3BhY2UoMC4wMDEsIDEuMCwgMTAwMCkKICAgIGJlc3QgPSAoZmxvYXQoImlu',
    'ZiIpLCBmbG9hdCgibmFuIikpCiAgICBmb3Igc3BhY2luZyBpbiBzcGFjaW5nczoKICAgICAgICBsb2dfbmV0ID0gKEZFQVRV',
    'UkVTICogbnAubG9nKG5wLmNlaWwoMiAvIHNwYWNpbmcpICsgMSkKICAgICAgICAgICAgICAgICAgICsgbnAubG9nKG5wLmNl',
    'aWwoMTAgLyBzcGFjaW5nKSArIDEpKQogICAgICAgIHJhZGl1cyA9IFNDT1JFX0JPVU5EICogbnAuc3FydCgobnAubG9nKDIg',
    'KiBob3Jpem9uIC8gZGVsdGFfbG9zcykgKyBsb2dfbmV0KSAvICgyICogbikpCiAgICAgICAgcmFkaXVzICs9IDggKiBzcGFj',
    'aW5nCiAgICAgICAgaWYgcmFkaXVzIDwgYmVzdFswXToKICAgICAgICAgICAgYmVzdCA9IChmbG9hdChyYWRpdXMpLCBmbG9h',
    'dChzcGFjaW5nKSkKICAgICMgVW5pZm9ybSBSYWRlbWFjaGVyIGJvdW5kIGZvciB0aGUgZWlnaHQgZWZmZWN0aXZlIHNjb3Jl',
    'IHBhcmFtZXRlcnMuCiAgICAjIEEgb25lLXNpZGVkIHN5bW1ldHJpemF0aW9uIHBsdXMgY29udHJhY3Rpb24gYm91bmRzIHRo',
    'ZSBleHBlY3RlZAogICAgIyBzdXByZW11bSBieSAyKig3KzUpL3NxcnQobik7IE1jRGlhcm1pZCBhbmQgYSB1bmlvbiBvdmVy',
    'IGJvdGggdGFpbHMKICAgICMgYW5kIGFsbCByb3VuZHMgY29udHJpYnV0ZSB0aGUgc2Vjb25kIHRlcm0uIFNlZSBBUFAtMDEg',
    'aW4gTWF0aC8uCiAgICBjb21wbGV4aXR5X3JhZGl1cyA9ICgyICogKEZFQVRVUkVTICsgNSkgLyBucC5zcXJ0KG4pCiAgICAg',
    'ICAgICAgICAgICAgICAgICAgICArIFNDT1JFX0JPVU5EICogbnAuc3FydChucC5sb2coMiAqIGhvcml6b24gLyBkZWx0YV9s',
    'b3NzKSAvICgyICogbikpKQogICAgcmV0dXJuIG1pbihTQ09SRV9CT1VORCwgYmVzdFswXSwgZmxvYXQoY29tcGxleGl0eV9y',
    'YWRpdXMpKSwgYmVzdFsxXQoKCmRlZiBnYXBfcmFkaXVzKG4wOiBpbnQsIG4xOiBpbnQsIGhvcml6b246IGludCwgZGVsdGFf',
    'Z2FwOiBmbG9hdCkgLT4gZmxvYXQ6CiAgICBpZiBtaW4objAsIG4xKSA8IDE6CiAgICAgICAgcmV0dXJuIEdBUF9CT1VORAog',
    'ICAgaWYgaG9yaXpvbiA8IDEgb3Igbm90IDAgPCBkZWx0YV9nYXAgPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImlu',
    'dmFsaWQgY2VydGlmaWNhdGUgcGFyYW1ldGVycyIpCiAgICBsb2dfdGVybSA9IG5wLmxvZyg0ICogRkVBVFVSRVMgKiBob3Jp',
    'em9uIC8gZGVsdGFfZ2FwKQogICAgZXBzMCA9IG5wLnNxcnQoMiAqIGxvZ190ZXJtIC8gbjApCiAgICBlcHMxID0gbnAuc3Fy',
    'dCgyICogbG9nX3Rlcm0gLyBuMSkKICAgIHJldHVybiBtaW4oR0FQX0JPVU5ELCBGRUFUVVJFUyAqIChlcHMwICsgZXBzMSkp',
    'CgoKZGVmIGNlcnRpZnkoZGF0YV9wYXRoOiBQYXRoLCByb3VuZHNfcGF0aDogUGF0aCwgb3V0cHV0OiBQYXRoLCBkZWx0YTog',
    'ZmxvYXQgPSAwLjA1KSAtPiBkaWN0OgogICAgaWYgbm90IDAgPCBkZWx0YSA8IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJv',
    'cigiZGVsdGEgbXVzdCBsaWUgaW4gKDAsMSkiKQogICAgd2l0aCBucC5sb2FkKGRhdGFfcGF0aCwgYWxsb3dfcGlja2xlPUZh',
    'bHNlKSBhcyBkYXRhOgogICAgICAgIGdyb3VwcyA9IG5wLmFzYXJyYXkoZGF0YVsiZ3JvdXBzIl0sIGludCkKICAgICAgICB3',
    'ZWlnaHRzID0gbnAuYXNhcnJheShkYXRhWyJ3ZWlnaHRzIl0sIGZsb2F0KQogICAgICAgIHNoYXBlID0gZGF0YVsiZmVhdHVy',
    'ZXMiXS5zaGFwZQogICAgaWYgbGVuKHNoYXBlKSAhPSAzIG9yIHNoYXBlWzJdICE9IDI0IG9yIGdyb3Vwcy5zaGFwZSAhPSBz',
    'aGFwZVs6Ml0gb3Igd2VpZ2h0cy5zaGFwZSAhPSAoc2hhcGVbMF0sIDMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRo',
    'aXMgY2VydGlmaWNhdGUgcmVxdWlyZXMgdGhlIEVYUC0xMCBzaWduZWQtcGFpciBmZWF0dXJlIG1hcCIpCiAgICB3aXRoIHJv',
    'dW5kc19wYXRoLm9wZW4obmV3bGluZT0iIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgaGFuZGxlOgogICAgICAgIHJvdW5kcyA9',
    'IGxpc3QoY3N2LkRpY3RSZWFkZXIoaGFuZGxlKSkKICAgIGhvcml6b24sIG4sIF8gPSBzaGFwZQogICAgaWYgbGVuKHJvdW5k',
    'cykgIT0gaG9yaXpvbjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJyb3VuZCBjb3VudCBtaXNtYXRjaCIpCiAgICByaXNr',
    'X3JhZGl1cywgc3BhY2luZyA9IGxvc3NfcmFkaXVzKG4sIGhvcml6b24sIGRlbHRhIC8gMikKICAgIHJvd3MgPSBbXQogICAg',
    'Zm9yIHQgaW4gcmFuZ2UoaG9yaXpvbik6CiAgICAgICAgbjAgPSBpbnQobnAuc3VtKGdyb3Vwc1t0XSA9PSAwKSkKICAgICAg',
    'ICBuMSA9IGludChucC5zdW0oZ3JvdXBzW3RdID09IDEpKQogICAgICAgIGdhcCA9IGdhcF9yYWRpdXMobjAsIG4xLCBob3Jp',
    'em9uLCBkZWx0YSAvIDIpCiAgICAgICAgY2VudGVyID0gbnAuYXNhcnJheShqc29uLmxvYWRzKHJvdW5kc1t0XVsibWlkcG9p',
    'bnRzIl0pLCBmbG9hdCkKICAgICAgICByYWRpdXMgPSBucC5hcnJheShbcmlza19yYWRpdXMsIGdhcCwgMC4wXSkKICAgICAg',
    'ICBpZiBjZW50ZXIuc2hhcGUgIT0gKDMsKSBvciBub3QgbnAuYWxsKG5wLmlzZmluaXRlKGNlbnRlcikpOgogICAgICAgICAg',
    'ICByYWlzZSBWYWx1ZUVycm9yKGYiaW52YWxpZCBvYmplY3RpdmUgY2VudGVyIGF0IGJhdGNoIHt0ICsgMX0iKQogICAgICAg',
    'IHJvd3MuYXBwZW5kKHsKICAgICAgICAgICAgImJhdGNoIjogdCArIDEsICJuMCI6IG4wLCAibjEiOiBuMSwKICAgICAgICAg',
    'ICAgIm9ic2VydmVkX2xvZ2xvc3NfY2VudGVyIjogZmxvYXQoY2VudGVyWzBdKSwKICAgICAgICAgICAgIm9ic2VydmVkX2dh',
    'cF9jZW50ZXIiOiBmbG9hdChjZW50ZXJbMV0pLAogICAgICAgICAgICAicmlza19yYWRpdXMiOiBmbG9hdChyYWRpdXNbMF0p',
    'LCAiZ2FwX3JhZGl1cyI6IGZsb2F0KHJhZGl1c1sxXSksCiAgICAgICAgICAgICJyZXNvdXJjZV9yYWRpdXMiOiAwLjAsCiAg',
    'ICAgICAgICAgICJ3ZWlnaHRlZF9oYWxmX3dpZHRoIjogZmxvYXQod2VpZ2h0c1t0XSBAIHJhZGl1cyksCiAgICAgICAgfSkK',
    'ICAgIG91dHB1dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIChvdXRwdXQgLyAicG9wdWxh',
    'dGlvbl9iYW5kX3dpZHRocy5jc3YiKS5vcGVuKCJ3IiwgbmV3bGluZT0iIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgaGFuZGxl',
    'OgogICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKGhhbmRsZSwgZmllbGRuYW1lcz1saXN0KHJvd3NbMF0pKQogICAg',
    'ICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgd3JpdGVyLndyaXRlcm93cyhyb3dzKQogICAgc3VtbWFyeSA9IHsK',
    'ICAgICAgICAic3RhdHVzIjogImNvbmRpdGlvbmFsX2NlcnRpZmljYXRlX29ubHkiLAogICAgICAgICJzYW1wbGluZ19hc3N1',
    'bXB0aW9uIjogImN1cnJlbnQgYmF0Y2ggSUlEIGNvbmRpdGlvbmFsIG9uIHBhc3Q7IHVudmVyaWZpZWQgZm9yIFVDSSBCYW5r',
    'IE1hcmtldGluZyIsCiAgICAgICAgInRhcmdldCI6ICJjb25kaXRpb25hbCBwb3B1bGF0aW9uIG9iamVjdGl2ZSBmdW5jdGlv',
    'bnMsIHVuaWZvcm1seSBvdmVyIGFsbCBib3ggYWN0aW9ucyIsCiAgICAgICAgInNpbXVsdGFuZW91c19jb25maWRlbmNlX2lm',
    'X2Fzc3VtcHRpb25faG9sZHMiOiAxIC0gZGVsdGEsCiAgICAgICAgImhvcml6b24iOiBob3Jpem9uLCAiYmF0Y2hfc2l6ZSI6',
    'IG4sCiAgICAgICAgInJpc2tfZ3JpZF9zcGFjaW5nIjogc3BhY2luZywKICAgICAgICAicmlza19yYWRpdXNfbWV0aG9kIjog',
    'Im1pbmltdW0gb2YgZ3JpZC1Ib2VmZmRpbmcgYW5kIHVuaWZvcm0gUmFkZW1hY2hlci1NY0RpYXJtaWQgYm91bmRzIiwKICAg',
    'ICAgICAicmlza19yYWRpdXMiOiByaXNrX3JhZGl1cywKICAgICAgICAibWVhbl9nYXBfcmFkaXVzIjogZmxvYXQobnAubWVh',
    'bihbcm93WyJnYXBfcmFkaXVzIl0gZm9yIHJvdyBpbiByb3dzXSkpLAogICAgICAgICJtYXhfZ2FwX3JhZGl1cyI6IGZsb2F0',
    'KG5wLm1heChbcm93WyJnYXBfcmFkaXVzIl0gZm9yIHJvdyBpbiByb3dzXSkpLAogICAgICAgICJtZWFuX29ic2VydmVkX2xv',
    'Z2xvc3NfY2VudGVyIjogZmxvYXQobnAubWVhbihbcm93WyJvYnNlcnZlZF9sb2dsb3NzX2NlbnRlciJdIGZvciByb3cgaW4g',
    'cm93c10pKSwKICAgICAgICAibWVhbl9vYnNlcnZlZF9nYXBfY2VudGVyIjogZmxvYXQobnAubWVhbihbcm93WyJvYnNlcnZl',
    'ZF9nYXBfY2VudGVyIl0gZm9yIHJvdyBpbiByb3dzXSkpLAogICAgICAgICJtZWFuX3dlaWdodGVkX2hhbGZfd2lkdGgiOiBm',
    'bG9hdChucC5tZWFuKFtyb3dbIndlaWdodGVkX2hhbGZfd2lkdGgiXSBmb3Igcm93IGluIHJvd3NdKSksCiAgICAgICAgIlRI',
    'TTAyX1VfVF91cHBlcl9ib3VuZCI6IGZsb2F0KDIgKiBucC5zdW0oW3Jvd1sid2VpZ2h0ZWRfaGFsZl93aWR0aCJdIGZvciBy',
    'b3cgaW4gcm93c10pKSwKICAgICAgICAiZW1waXJpY2FsX3BvcHVsYXRpb25fY292ZXJhZ2VfbWVhc3VyYWJsZSI6IEZhbHNl',
    'LAogICAgfQogICAgaWYgbnAuYWxsY2xvc2Uod2VpZ2h0cywgWzAuOCwgMC4xNSwgMC4wNV0pOgogICAgICAgIGdyYWRpZW50',
    'X2JvdW5kID0gMC44NSAqIG1hdGguc3FydCgyNCkgKyAwLjE1ICogMiAqIG1hdGguc3FydCgxNCkKICAgICAgICBvZ2Rfc3Rh',
    'dGljX3BhcnQgPSAyNCAvICgyICogMC4wNSkgKyAwLjA1ICogZ3JhZGllbnRfYm91bmQqKjIgKiBob3Jpem9uIC8gMgogICAg',
    'ICAgIHN1bW1hcnlbInN0YXRpY19jb21wYXJhdG9yX2VudmVsb3BlX2F0X2V0YV8wLjA1Il0gPSB7CiAgICAgICAgICAgICJn',
    'cmFkaWVudF9ub3JtX2JvdW5kIjogZ3JhZGllbnRfYm91bmQsCiAgICAgICAgICAgICJvcHRpbWl6YXRpb25fcGFydCI6IG9n',
    'ZF9zdGF0aWNfcGFydCwKICAgICAgICAgICAgInRvdGFsX3dpdGhfVV9UIjogb2dkX3N0YXRpY19wYXJ0ICsgc3VtbWFyeVsi',
    'VEhNMDJfVV9UX3VwcGVyX2JvdW5kIl0sCiAgICAgICAgICAgICJpbnRlcnByZXRhdGlvbiI6ICJjb25kaXRpb25hbCB3b3Jz',
    'dC1jYXNlIGJvdW5kLCBub3QgbWVhc3VyZWQgcmVncmV0IiwKICAgICAgICB9CiAgICAob3V0cHV0IC8gInBvcHVsYXRpb25f',
    'YmFuZF9zdW1tYXJ5Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoc3VtbWFyeSwgaW5kZW50PTIpLCBlbmNvZGluZz0i',
    'dXRmLTgiKQogICAgcmV0dXJuIHN1bW1hcnkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcGFyc2VyID0gYXJn',
    'cGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhIiwgdHlwZT1QYXRoLCByZXF1',
    'aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1yb3VuZHMiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUp',
    'CiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNl',
    'ci5hZGRfYXJndW1lbnQoIi0tZGVsdGEiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuMDUpCiAgICBhcmdzID0gcGFyc2VyLnBh',
    'cnNlX2FyZ3MoKQogICAgcHJpbnQoanNvbi5kdW1wcyhjZXJ0aWZ5KGFyZ3MuZGF0YSwgYXJncy5yb3VuZHMsIGFyZ3Mub3V0',
    'cHV0LCBhcmdzLmRlbHRhKSwgaW5kZW50PTIpKQo=',
]))
PROOF_TEXT = base64.b64decode("".join([
    'IyBDb25kaXRpb25hbCBwb3B1bGF0aW9uIGludGVydmFscyBmb3IgRVhQLTEwCgoqKlN0YXR1czogY29uZGl0aW9uYWwgcHJv',
    'cG9zaXRpb24sIHByb3ZlZC4qKiBUaGlzIGlzIGEgc2VwYXJhdGUgc3RvY2hhc3RpYwppbnRlcnByZXRhdGlvbiBvZiBFWFAt',
    'MTAuIFRoZSBwdWJsaXNoZWQgZGF0ZSBvcmRlciBvZiBVQ0kgQmFuayBNYXJrZXRpbmcgZG9lcwpub3QgdmVyaWZ5IGl0cyBz',
    'YW1wbGluZyBhc3N1bXB0aW9uLgoKIyMgVGFyZ2V0IGFuZCBhc3N1bXB0aW9ucwoKQXQgcm91bmQgYHRgLCBsZXQgYEhfe3Qt',
    'MX1gIGNvbnRhaW4gZXZlcnkgZWFybGllciBiYXRjaCBhbmQgdGhlIHBhc3Qtb25seQpmZWF0dXJlIHRyYW5zZm9ybS4gQ29u',
    'ZGl0aW9uYWwgb24gYEhfe3QtMX1gLCB0aGUgMjU2IG9ic2VydmF0aW9ucwpgKFNfaSxZX2ksR19pKWAgaW4gdGhlIGN1cnJl',
    'bnQgYmF0Y2ggYXJlIGluZGVwZW5kZW50IGFuZCBpZGVudGljYWxseQpkaXN0cmlidXRlZCBmcm9tIGEgZGlzdHJpYnV0aW9u',
    'IGBQX3RgLiBUaGUgZGlzdHJpYnV0aW9uIG1heSBjaGFuZ2Ugd2l0aCBgdGAKYW5kIG1heSBkZXBlbmQgb24gYEhfe3QtMX1g',
    'LiBBc3N1bWUgYFPiiIhbLTEsMV1eN2AsIGBZ4oiIey0xLCsxfWAgYW5kIGJvdGggYWdlCmdyb3VwcyBoYXZlIHBvc2l0aXZl',
    'IHByb2JhYmlsaXR5LiBUaGVzZSBhc3N1bXB0aW9ucyBjb25jZXJuIHRoZSB0cmFuc2Zvcm1lZApzZXZlbiBwcmUtY2FsbCB2',
    'YXJpYWJsZXMsIG5vdCB0aGUgc2lnbmVkLXBhaXIgcmVwcmVzZW50YXRpb24uCgpGb3IgYSBib3ggYWN0aW9uIGB34oiIWzAs',
    'MV1eMjRgLCB0aGUgZXhpc3RpbmcgZmVhdHVyZSBtYXAgaW5kdWNlcyBlZmZlY3RpdmUKY29lZmZpY2llbnRzIGDOuCh3KeKI',
    'iFstMSwxXV43YCBhbmQgaW50ZXJjZXB0IGBiKHcp4oiIWy01LDVdYCwgd2l0aCBzY29yZQpgel93KFMpPc64KHcpwrdTK2Io',
    'dylgLiBIZW5jZSBgfHpfdyhTKXziiaQxMmAuIERlZmluZSBwb3B1bGF0aW9uIG9iamVjdGl2ZXMKCmBoX3QsMCh3KT1FX3tQ',
    'X3R9W2xvZygxK2V4cCgtWSB6X3coUykpKV1gLAoKYGhfdCwxKHcpPXxFX3tQX3R9W3pfdyhTKXxHPTBdLUVfe1BfdH1bel93',
    'KFMpfEc9MV18YCwgYW5kCgpgaF90LDIodyk9fHx3fHzCsi8yYC4KCkxldCBgY190LGpgIGJlIHRoZSBjb3JyZXNwb25kaW5n',
    'IGN1cnJlbnQtYmF0Y2ggb2JqZWN0aXZlcy4gQm90aCBgaF90LGpgIGFuZApgY190LGpgIGFyZSBjb252ZXggaW4gYHdgLiBU',
    'aGUgaW50ZXJjZXB0IGNhbmNlbHMgZnJvbSB0aGUgZ3JvdXAgZ2FwLiBUaGUKY3VycmVudCBiYXRjaCwgaXRzIG9iamVjdGl2',
    'ZXMsIGFuZCBpbnRlcnZhbCBlbmRwb2ludHMgYmVjb21lIGF2YWlsYWJsZSBvbmx5CmFmdGVyIHRoZSBsZWFybmVyIGhhcyBz',
    'ZWxlY3RlZCBgd190YDsgdGhpcyBtYXRjaGVzIHRoZSBwb3N0LWRlY2lzaW9uIGZlZWRiYWNrCnByb3RvY29sLiBUaGUgbGVh',
    'cm5lcidzIGFjdGlvbiBjYW4gZGVwZW5kIG9uIHRoZSBwYXN0IGJ1dCBub3QgdGhlIGN1cnJlbnQKYmF0Y2guCgojIyBTaW11',
    'bHRhbmVvdXMgaW50ZXJ2YWwgY29uc3RydWN0aW9uCgpGaXggYM604oiIKDAsMSlgLCBob3Jpem9uIGBUYCwgYW5kIHNwbGl0',
    'IGDOtD3OtF8wK860XzFgLiBTZWxlY3QgYSBkZXRlcm1pbmlzdGljCmdyaWQgc3BhY2luZyBgYeKIiCgwLDFdYCB1c2luZyBv',
    'bmx5IHB1YmxpYyBgbixULM60YC4gQSBjb29yZGluYXRlIGdyaWQgb2YKYFstMSwxXV43w5dbLTUsNV1gIHdpdGggbWF4aW11',
    'bSBzcGFjaW5nIGBhYCBjb250YWlucyBhdCBtb3N0CgpgTihhKT0oY2VpbCgyL2EpKzEpXjcgKGNlaWwoMTAvYSkrMSlgIHBv',
    'aW50cy4KCkxldCBgbl90YCBiZSB0aGUgYmF0Y2ggc2l6ZSBhbmQgYG5fdGdgIGl0cyBncm91cCBjb3VudHMuIERlZmluZQoK',
    'YHJfdCwwKGEpPW1pbnsxMiwgMTIgc3FydChsb2coMiBOKGEpIFQvzrRfMCkvKDJuX3QpKSs4YSwKMjQvc3FydChuX3QpKzEy',
    'IHNxcnQobG9nKDJUL860XzApLygybl90KSl9YCwKCmBlX3RnPXNxcnQoMiBsb2coMjhUL860XzEpL25fdGcpYCBmb3IgYG5f',
    'dGc+MGAsCgpgcl90LDE9bWluezE0LCA3KGVfdDArZV90MSl9YCwgYW5kIGByX3QsMj0wYC4KCklmIGEgZ3JvdXAgaXMgYWJz',
    'ZW50LCBzZXQgYGNfdCwxPTBgIGFuZCBgcl90LDE9MTRgIGZvciB0aGF0IHJvdW5kLiBTZXQKYExfdCxqKHcpPWNfdCxqKHcp',
    'LXJfdCxqYCBhbmQgYFVfdCxqKHcpPWNfdCxqKHcpK3JfdCxqYC4gVW5kZXIgdGhlIGFzc3VtcHRpb25zCmFib3ZlLCB3aXRo',
    'IHByb2JhYmlsaXR5IGF0IGxlYXN0IGAxLc60YCwgKipzaW11bHRhbmVvdXNseSBmb3IgZXZlcnkgcm91bmQsCm9iamVjdGl2',
    'ZSwgYW5kIGFjdGlvbioqLCBgTF90LGoodyniiaRoX3Qsaih3KeKJpFVfdCxqKHcpYC4gVGhlIGNvbnN0YW50IHJhZGlpCnBy',
    'ZXNlcnZlIGNvbnZleGl0eSBvZiBib3RoIGVuZHBvaW50cyBhbmQgbGVhdmUgdGhlIG1pZHBvaW50IGdyYWRpZW50cwp1bmNo',
    'YW5nZWQuIFRoZSBrbm93biByZXNvdXJjZSBvYmplY3RpdmUgaGFzIGV4YWN0IHplcm8gd2lkdGguCgojIyMgUHJvb2YKCkZv',
    'ciBldmVyeSBmaXhlZCBncmlkIGFjdGlvbiwgbG9naXN0aWMgbG9zcyBoYXMgYSByYW5nZSBvZiBsZW5ndGggMTI6IGl0cwpt',
    'YXJnaW4gbGllcyBpbiBgWy0xMiwxMl1gLCBhbmQKYGxvZygxK2V4cCgxMikpLWxvZygxK2V4cCgtMTIpKT0xMmAuIENvbmRp',
    'dGlvbmFsIEhvZWZmZGluZyBnaXZlcwpgUHIofGNfdCwwLWhfdCwwfD5xIHwgSF97dC0xfSniiaQyIGV4cCgtMm5fdCBxwrIv',
    'MTLCsilgIGF0IHRoYXQgYWN0aW9uLgpBcHBseSBhIHVuaW9uIGJvdW5kIHRvIHRoZSBgTihhKWAgZ3JpZCBwb2ludHMgYW5k',
    'IGBUYCByb3VuZHMsIGFsbG9jYXRpbmcKYM60XzBgLiBMb2dpc3RpYyBsb3NzIGlzIDEtTGlwc2NoaXR6IGluIHNjb3JlOyB0',
    'aGUgZW1waXJpY2FsIGFuZCBwb3B1bGF0aW9uCnJpc2tzIGFyZSBlYWNoIDEtTGlwc2NoaXR6IGluIGBsXzFgIGRpc3RhbmNl',
    'IG9mIGAozrgsYilgLiBBIG5lYXJlc3QgZ3JpZCBwb2ludAppcyB3aXRoaW4gYDRhYCBpbiB0aGF0IGRpc3RhbmNlLCBzbyB0',
    'aGUgZGlzY3JlcGFuY3kgYXQgYW4gYXJiaXRyYXJ5IGFjdGlvbgppcyB3aXRoaW4gYDhhYCBvZiB0aGUgZ3JpZCBkaXNjcmVw',
    'YW5jeS4gU2luY2UgdHdvIGV4cGVjdGF0aW9ucyBvZiB2YXJpYWJsZXMKaW4gdGhlIHNhbWUgbGVuZ3RoLTEyIHJhbmdlIGNh',
    'biBkaWZmZXIgYnkgYXQgbW9zdCAxMiwgdGhlIG1pbmltdW0gd2l0aCAxMgphbHNvIHByZXNlcnZlcyB0aGUgYm91bmQuCgpG',
    'b3IgdGhlIHRoaXJkIGNhbmRpZGF0ZSByYWRpdXMsIGNvbnNpZGVyIGVpdGhlciBzaWduIG9mIHRoZSByaXNrCmRldmlhdGlv',
    'bi4gT25lLXNpZGVkIHN5bW1ldHJpemF0aW9uIGJvdW5kcyB0aGUgZXhwZWN0ZWQgc3VwcmVtdW0gYnkgdHdpY2UKdGhlIFJh',
    'ZGVtYWNoZXIgY29tcGxleGl0eSBvZiB0aGUgbG9zcyBjbGFzcy4gVGhlIG5vLWFic29sdXRlLXZhbHVlCmNvbnRyYWN0aW9u',
    'IGZvcm0gaXMgc3RhdGVkIGFuZCBwcm92ZWQgaW4gW01vaHJpIGFuZCBNdcOxb3ogTWVkaW5hLApKTUxSIDIwMTYsIExlbW1h',
    'IDE0XShodHRwczovL2ptbHIyMDIwLmNzYWlsLm1pdC5lZHUvcGFwZXJzL3ZvbHVtZTE3LzE0LTQ5OS8xNC00OTkucGRmKS4K',
    'VGhlIGNvbnRyYWN0aW9uIGluZXF1YWxpdHkKYXBwbGllcyBiZWNhdXNlIGB24oambG9nKDErZXhwKC1ZdikpLWxvZyAyYCBp',
    'cyAxLUxpcHNjaGl0eiBhbmQgemVybyBhdAp6ZXJvLiBDb25kaXRpb25hbCBvbiB0aGUgb2JzZXJ2YXRpb25zLCB0aGUgc2Nv',
    'cmUgY2xhc3MgaGFzIGNvbXBsZXhpdHkKYXQgbW9zdCBgKDcrNSkvc3FydChuX3QpPTEyL3NxcnQobl90KWA6IGVhY2ggb2Yg',
    'dGhlIHNldmVuIGNvb3JkaW5hdGUKUmFkZW1hY2hlciBzdW1zIGFuZCB0aGUgaW50ZXJjZXB0IHN1bSBoYXMgZXhwZWN0ZWQg',
    'YWJzb2x1dGUgdmFsdWUgYXQKbW9zdCBgc3FydChuX3QpYC4gVGh1cyB0aGUgZXhwZWN0ZWQgb25lLXNpZGVkIHJpc2sgc3Vw',
    'cmVtdW0gaXMgYXQgbW9zdApgMjQvc3FydChuX3QpYC4gUmVwbGFjaW5nIG9uZSBvYnNlcnZhdGlvbiBjaGFuZ2VzIGVpdGhl',
    'ciBvbmUtc2lkZWQKc3VwcmVtdW0gYnkgYXQgbW9zdCBgMTIvbl90YC4gTWNEaWFybWlkJ3MgaW5lcXVhbGl0eSBhbmQgYSB1',
    'bmlvbiBib3VuZApvdmVyIGJvdGggc2lnbnMgYW5kIGBUYCByb3VuZHMgYWRkCmAxMiBzcXJ0KGxvZygyVC/OtF8wKS8oMm5f',
    'dCkpYC4gVGhlIGNvbnN0cnVjdGlvbiBzZWxlY3RzIHRoZSBzbWFsbGVzdApjYW5kaWRhdGUgZnJvbSBwdWJsaWMgYG5fdCxU',
    'LM60XzBgIGJlZm9yZSBvYnNlcnZpbmcgdGhlIGN1cnJlbnQgYmF0Y2gsCnNvIG9ubHkgdGhhdCBjYW5kaWRhdGUncyBmYWls',
    'dXJlIHByb2JhYmlsaXR5IGlzIGNoYXJnZWQuIElmIGJhdGNoIHNpemUKaXMgbm90IGZpeGVkIGluIGFkdmFuY2UsIGl0cyB2',
    'YWx1ZSBtdXN0IGJlIHByZWRpY3RhYmxlIGJlZm9yZSB0aGUgYmF0Y2guCgpDb25kaXRpb25hbCBvbiBhIGJhdGNoJ3MgcG9z',
    'aXRpdmUgZ3JvdXAgY291bnRzLCB0aGUgb2JzZXJ2YXRpb25zIGluIGVhY2gKZ3JvdXAgYXJlIGlpZCBmcm9tIGBQX3QoU3xH',
    'PWcpYC4gRWFjaCBjb29yZGluYXRlIGxpZXMgaW4gYFstMSwxXWAuCkhvZWZmZGluZyBhbmQgYSB1bmlvbiBib3VuZCBvdmVy',
    'IHR3byBncm91cHMsIHNldmVuIGNvb3JkaW5hdGVzLCBhbmQgYFRgCnJvdW5kcyBzaG93IHRoYXQgYWxsIGdyb3VwLWNvb3Jk',
    'aW5hdGUgbWVhbiBlcnJvcnMgYXJlIGF0IG1vc3QgYGVfdGdgIHdpdGgKcHJvYmFiaWxpdHkgYXQgbGVhc3QgYDEtzrRfMWAu',
    'IEZvciBldmVyeSBgfM64X2p84omkMWAsIHRoZSBkaWZmZXJlbmNlIG9mIHRoZQpwb3B1bGF0aW9uIGFuZCBlbXBpcmljYWwg',
    'c2lnbmVkIGdyb3VwIGNvbnRyYXN0cyBpcyBhdCBtb3N0CmA3KGVfdDArZV90MSlgLiBUaGUgYWJzb2x1dGUtdmFsdWUgbWFw',
    'IGlzIDEtTGlwc2NoaXR6LCBzbyB0aGlzIGFsc28gYm91bmRzCnRoZSBnYXAtb2JqZWN0aXZlIGRpc2NyZXBhbmN5LiBCb3Ro',
    'IGdhcHMgbGllIGluIGBbMCwxNF1gLCBhbGxvd2luZyB0aGUgY2FwCmF0IDE0LiBUaGUgZW1wdHktZ3JvdXAgZmFsbGJhY2sg',
    'Y292ZXJzIHRoYXQgY2FzZSBkZXRlcm1pbmlzdGljYWxseS4gVGhlCnJlc291cmNlIG9iamVjdGl2ZSBpcyBpZGVudGljYWwg',
    'aW4gc2FtcGxlIGFuZCBwb3B1bGF0aW9uLiBGaW5hbGx5LCB1bmlvbgp0aGUgcmlzayBhbmQgZ3JvdXAgZXZlbnRzLiBUaGUg',
    'YXJndW1lbnQgaXMgY29uZGl0aW9uYWwgYXQgZWFjaCByb3VuZCBhbmQKdGhlbiB1c2VzIHRoZSB0b3dlciBwcm9wZXJ0eTsg',
    'aXQgZG9lcyBub3QgcmVxdWlyZSBzdGF0aW9uYXJ5IGBQX3RgLiDilqEKCiMjIEludGVycHJldGF0aW9uCgpUaGlzIGNvbnN0',
    'cnVjdGlvbiBzdXBwbGllcyBBNCB3aXRoIGNvbmZpZGVuY2UgYDEtzrRgIHVuZGVyIGl0cyBzYW1wbGluZwphc3N1bXB0aW9u',
    'LCB1bmlmb3JtbHkgb3ZlciBoaW5kc2lnaHQgY29tcGFyYXRvciBhY3Rpb25zLiBJdCBhbHNvIHN1cHBsaWVzCmNvbnZleCBv',
    'cmRlcmVkIGVuZHBvaW50cyBmb3IgQTMgYW5kIHBvc3QtZGVjaXNpb24gbWlkcG9pbnQgZ3JhZGllbnRzIGZvciBBNS4KVEhN',
    'LTAyIHRoZW4gYXBwbGllcyBvbiB0aGUgc2ltdWx0YW5lb3VzLWNvbnRhaW5tZW50IGV2ZW50LCBzdWJqZWN0IHRvIGl0cwpy',
    'ZW1haW5pbmcgYXNzdW1wdGlvbnMuIFRoaXMgaXMgYSAqKmhpZ2gtcHJvYmFiaWxpdHkgY29uZGl0aW9uYWwgdGhlb3JlbSoq',
    'LCBub3QKYW4gdW5jb25kaXRpb25hbCBjbGFpbSBhYm91dCB0aGUgVUNJIHN0cmVhbS4gSWYgb2JzZXJ2YXRpb25zIHdpdGhp',
    'biBhIGJhdGNoCmFyZSBkZXBlbmRlbnQsIG9yIGlmIHRoZSByZXBvcnRlZCByb3cgb3JkZXIgZG9lcyBub3QgY29ycmVzcG9u',
    'ZCB0byBkcmF3cwpmcm9tIGEgc2luZ2xlIGBQX3RgLCB0aGlzIHByb29mIGdpdmVzIG5vIGNvdmVyYWdlIGd1YXJhbnRlZS4g',
    'V2lkZSBpbnRlcnZhbHMKbWF5IG1ha2UgdGhlIHJlc3VsdGluZyByZWdyZXQgZW52ZWxvcGUgdW5pbmZvcm1hdGl2ZS4KCkZv',
    'ciB0aGUgRVhQLTEwIHdlaWdodHMgYCgwLjgsMC4xNSwwLjA1KWAsIGEgdmFsaWQgbWlkcG9pbnQgc3ViZ3JhZGllbnQgbm9y',
    'bQpib3VuZCBpcyBgRz0wLjg1IHNxcnQoMjQpKzAuMzAgc3FydCgxNClgOiBsb2dpc3RpYyBhbmQgcmVzb3VyY2UgZ3JhZGll',
    'bnRzCmVhY2ggaGF2ZSBub3JtIGF0IG1vc3QgYHNxcnQoMjQpYDsgdGhlIGFnZS1nYXAgc3ViZ3JhZGllbnQgaGFzIG5vcm0g',
    'YXQgbW9zdApgMiBzcXJ0KDE0KWAgYmVjYXVzZSB0aGUgdGVuIGJpYXMgY29vcmRpbmF0ZXMgY2FuY2VsLiBUaGUgYm94IGRp',
    'YW1ldGVyIGlzCmBEPXNxcnQoMjQpYC4gVGhlc2UgY29uc3RhbnRzIHBlcm1pdCBhIGZ1bGx5IG51bWVyaWMgVEhNLTAyIGVu',
    'dmVsb3BlIG9uY2UKdGhlIGNvbmRpdGlvbmFsIHJhZGlpIGFyZSBjb21wdXRlZC4gVGhleSBkbyBub3QgbWFrZSBpdCB0aWdo',
    'dC4K',
]))
(CODE / "experiments" / "certify_bank_population.py").write_bytes(CERTIFIER_SOURCE)
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "streaming_population_intervals.md").write_bytes(PROOF_TEXT)
if RUN_OK:
    command = [sys.executable, str(CODE / "experiments" / "certify_bank_population.py"),
               "--data", str(DATA), "--rounds", str(ROUND_PATH),
               "--output", str(RESULTS / "population_certificate"), "--delta", "0.05"]
    outcome = subprocess.run(command, cwd=CODE, text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(outcome.stdout[-6000:])
    if outcome.returncode == 0:
        CERTIFICATE = json.loads((RESULTS / "population_certificate" /
                                  "population_band_summary.json").read_text(encoding="utf-8"))
        record("population_certificate", "passed", "conditional widths saved; IID assumption unverified")
    else:
        RUN_OK = False
        record("population_certificate", "failed", f"exit={outcome.returncode}")
else:
    record("population_certificate", "skipped", "earlier gate failed")


## 5. Check sharper bands under designed IID feedback


In [ ]:
CONTROLLED_SOURCE = base64.b64decode("".join([
    'IiIiVGVzdCBBUFAtMDEgd2l0aCBrbm93biBmaW5pdGUtd2luZG93IHBvcHVsYXRpb24gb2JqZWN0aXZlcy4KCkVhY2ggb3Jp',
    'Z2luYWwgY2hyb25vbG9naWNhbCBiYXRjaCBpcyBhIGZpeGVkIGZpbml0ZSBwb3B1bGF0aW9uLiBGZWVkYmFjayBpcwpzYW1w',
    'bGVkIElJRCB3aXRoIHJlcGxhY2VtZW50IGZyb20gdGhhdCBiYXRjaC4gVGhpcyBkZXNpZ25lZCBleHBlcmltZW50IG1lZXRz',
    'CkFQUC0wMSdzIHdpdGhpbi1yb3VuZCBzYW1wbGluZyBhc3N1bXB0aW9uIGNvbmRpdGlvbmFsIG9uIHRoZSBmaXhlZCBwb29s',
    'cy4KSXQgaXMgbm90IGFuIG9ic2VydmF0aW9uYWwgY292ZXJhZ2UgY2xhaW0gZm9yIHRoZSBvcmlnaW5hbCBVQ0kgc3RyZWFt',
    'LgoiIiIKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgY3N2CmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApp',
    'bXBvcnQgc3lzCgppbXBvcnQgbnVtcHkgYXMgbnAKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVz',
    'b2x2ZSgpLnBhcmVudHNbMV0gLyAic3JjIikpCmZyb20gaW50ZXJ2YWxfcGFyZXRvLmFsZ29yaXRobXMgaW1wb3J0IE1pZHBv',
    'aW50T0dECmZyb20gaW50ZXJ2YWxfcGFyZXRvLmFwcGxpY2F0aW9ucy5zdHJlYW1pbmcgaW1wb3J0IENsYXNzaWZpY2F0aW9u',
    'QmF0Y2gsIG9iamVjdGl2ZV9hbmRfc3ViZ3JhZGllbnRzCmZyb20gY2VydGlmeV9iYW5rX3BvcHVsYXRpb24gaW1wb3J0IGdh',
    'cF9yYWRpdXMsIGxvc3NfcmFkaXVzCgoKZGVmIHJ1bihkYXRhX3BhdGg6IFBhdGgsIG91dHB1dDogUGF0aCwgc2VlZHM6IHR1',
    'cGxlW2ludCwgLi4uXSA9IHR1cGxlKHJhbmdlKDEwKSksCiAgICAgICAgZXRhOiBmbG9hdCA9IDAuMDUsIGRlbHRhOiBmbG9h',
    'dCA9IDAuMDUpIC0+IGRpY3Q6CiAgICBpZiBub3Qgc2VlZHMgb3IgbGVuKHNldChzZWVkcykpICE9IGxlbihzZWVkcykgb3Ig',
    'ZXRhIDw9IDAgb3Igbm90IDAgPCBkZWx0YSA8IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW52YWxpZCBzZWVkcywg',
    'ZXRhLCBvciBkZWx0YSIpCiAgICB3aXRoIG5wLmxvYWQoZGF0YV9wYXRoLCBhbGxvd19waWNrbGU9RmFsc2UpIGFzIGRhdGE6',
    'CiAgICAgICAgeCA9IG5wLmFzYXJyYXkoZGF0YVsiZmVhdHVyZXMiXSwgZmxvYXQpCiAgICAgICAgeSA9IG5wLmFzYXJyYXko',
    'ZGF0YVsibGFiZWxzIl0sIGZsb2F0KQogICAgICAgIGcgPSBucC5hc2FycmF5KGRhdGFbImdyb3VwcyJdLCBpbnQpCiAgICAg',
    'ICAgd2VpZ2h0cyA9IG5wLmFzYXJyYXkoZGF0YVsid2VpZ2h0cyJdLCBmbG9hdCkKICAgIGlmIHgubmRpbSAhPSAzIG9yIHgu',
    'c2hhcGVbMl0gIT0gMjQgb3IgeS5zaGFwZSAhPSB4LnNoYXBlWzoyXSBvciBnLnNoYXBlICE9IHkuc2hhcGU6CiAgICAgICAg',
    'cmFpc2UgVmFsdWVFcnJvcigiRVhQLTEwIGFycmF5cyByZXF1aXJlZCIpCiAgICBob3Jpem9uLCBuLCBkaW1lbnNpb24gPSB4',
    'LnNoYXBlCiAgICByaXNrX3JhZGl1cywgc3BhY2luZyA9IGxvc3NfcmFkaXVzKG4sIGhvcml6b24sIGRlbHRhIC8gMikKICAg',
    'IHJvd3MgPSBbXQogICAgZm9yIHNlZWQgaW4gc2VlZHM6CiAgICAgICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNl',
    'ZWQpCiAgICAgICAgbGVhcm5lciA9IE1pZHBvaW50T0dEKGRpbWVuc2lvbiwgZXRhKQogICAgICAgIGZvciB0IGluIHJhbmdl',
    'KGhvcml6b24pOgogICAgICAgICAgICBhY3Rpb24gPSBsZWFybmVyLmFjdCgpCiAgICAgICAgICAgIHBvb2wgPSBDbGFzc2lm',
    'aWNhdGlvbkJhdGNoKHhbdF0sIHlbdF0sIGdbdF0pCiAgICAgICAgICAgIGRyYXcgPSBybmcuaW50ZWdlcnMoMCwgbiwgc2l6',
    'ZT1uKQogICAgICAgICAgICBpZiBub3QgbnAuYW55KGdbdCwgZHJhd10gPT0gMCkgb3Igbm90IG5wLmFueShnW3QsIGRyYXdd',
    'ID09IDEpOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJSYXJlIGVtcHR5IHNhbXBsZWQgYWdlIGdyb3Vw',
    'OyBjaG9vc2UgYSBuZXcgcHJlZGVjbGFyZWQgc2VlZCBzZXQiKQogICAgICAgICAgICBzYW1wbGUgPSBDbGFzc2lmaWNhdGlv',
    'bkJhdGNoKHhbdCwgZHJhd10sIHlbdCwgZHJhd10sIGdbdCwgZHJhd10pCiAgICAgICAgICAgIHBvcHVsYXRpb24sIF8gPSBv',
    'YmplY3RpdmVfYW5kX3N1YmdyYWRpZW50cyhwb29sLCBhY3Rpb24pCiAgICAgICAgICAgIG9ic2VydmVkLCBncmFkaWVudHMg',
    'PSBvYmplY3RpdmVfYW5kX3N1YmdyYWRpZW50cyhzYW1wbGUsIGFjdGlvbikKICAgICAgICAgICAgbjAgPSBpbnQobnAuc3Vt',
    'KHNhbXBsZS5ncm91cHMgPT0gMCkpCiAgICAgICAgICAgIG4xID0gbiAtIG4wCiAgICAgICAgICAgIHJhZGlpID0gbnAuYXJy',
    'YXkoW3Jpc2tfcmFkaXVzLCBnYXBfcmFkaXVzKG4wLCBuMSwgaG9yaXpvbiwgZGVsdGEgLyAyKSwgMC4wXSkKICAgICAgICAg',
    'ICAgZXJyb3IgPSBucC5hYnMob2JzZXJ2ZWQgLSBwb3B1bGF0aW9uKQogICAgICAgICAgICByb3dzLmFwcGVuZCh7CiAgICAg',
    'ICAgICAgICAgICAic2VlZCI6IHNlZWQsICJiYXRjaCI6IHQgKyAxLCAibjAiOiBuMCwgIm4xIjogbjEsCiAgICAgICAgICAg',
    'ICAgICAicG9wdWxhdGlvbl9sb2dsb3NzIjogZmxvYXQocG9wdWxhdGlvblswXSksCiAgICAgICAgICAgICAgICAic2FtcGxl',
    'X2xvZ2xvc3MiOiBmbG9hdChvYnNlcnZlZFswXSksCiAgICAgICAgICAgICAgICAicG9wdWxhdGlvbl9nYXAiOiBmbG9hdChw',
    'b3B1bGF0aW9uWzFdKSwKICAgICAgICAgICAgICAgICJzYW1wbGVfZ2FwIjogZmxvYXQob2JzZXJ2ZWRbMV0pLAogICAgICAg',
    'ICAgICAgICAgInJpc2tfZXJyb3IiOiBmbG9hdChlcnJvclswXSksICJnYXBfZXJyb3IiOiBmbG9hdChlcnJvclsxXSksCiAg',
    'ICAgICAgICAgICAgICAicmlza19yYWRpdXMiOiBmbG9hdChyYWRpaVswXSksICJnYXBfcmFkaXVzIjogZmxvYXQocmFkaWlb',
    'MV0pLAogICAgICAgICAgICAgICAgInJpc2tfY292ZXJlZCI6IGludChlcnJvclswXSA8PSByYWRpaVswXSArIDFlLTEyKSwK',
    'ICAgICAgICAgICAgICAgICJnYXBfY292ZXJlZCI6IGludChlcnJvclsxXSA8PSByYWRpaVsxXSArIDFlLTEyKSwKICAgICAg',
    'ICAgICAgICAgICJmaXhlZF9yaXNrX2NvdmVyZWQiOiBpbnQoZXJyb3JbMF0gPD0gMC4wNSArIDFlLTEyKSwKICAgICAgICAg',
    'ICAgICAgICJmaXhlZF9nYXBfY292ZXJlZCI6IGludChlcnJvclsxXSA8PSAwLjA1ICsgMWUtMTIpLAogICAgICAgICAgICB9',
    'KQogICAgICAgICAgICBsZWFybmVyLm9ic2VydmVfZ3JhZGllbnQod2VpZ2h0c1t0XSBAIGdyYWRpZW50cykKICAgIG91dHB1',
    'dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIChvdXRwdXQgLyAiY29udHJvbGxlZF9yZXNh',
    'bXBsaW5nX3JvdW5kcy5jc3YiKS5vcGVuKCJ3IiwgbmV3bGluZT0iIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgaGFuZGxlOgog',
    'ICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKGhhbmRsZSwgZmllbGRuYW1lcz1saXN0KHJvd3NbMF0pKQogICAgICAg',
    'IHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgd3JpdGVyLndyaXRlcm93cyhyb3dzKQogICAgcGVyX3NlZWQgPSBbXQog',
    'ICAgZm9yIHNlZWQgaW4gc2VlZHM6CiAgICAgICAgc3Vic2V0ID0gW3JvdyBmb3Igcm93IGluIHJvd3MgaWYgcm93WyJzZWVk',
    'Il0gPT0gc2VlZF0KICAgICAgICBwZXJfc2VlZC5hcHBlbmQoewogICAgICAgICAgICAic2VlZCI6IHNlZWQsCiAgICAgICAg',
    'ICAgICJhbGxfcGxheWVkX2FjdGlvbnNfY292ZXJlZCI6IGFsbChyb3dbInJpc2tfY292ZXJlZCJdIGFuZCByb3dbImdhcF9j',
    'b3ZlcmVkIl0gZm9yIHJvdyBpbiBzdWJzZXQpLAogICAgICAgICAgICAicmlza19jb3ZlcmFnZSI6IGZsb2F0KG5wLm1lYW4o',
    'W3Jvd1sicmlza19jb3ZlcmVkIl0gZm9yIHJvdyBpbiBzdWJzZXRdKSksCiAgICAgICAgICAgICJnYXBfY292ZXJhZ2UiOiBm',
    'bG9hdChucC5tZWFuKFtyb3dbImdhcF9jb3ZlcmVkIl0gZm9yIHJvdyBpbiBzdWJzZXRdKSksCiAgICAgICAgICAgICJmaXhl',
    'ZF9yaXNrX2NvdmVyYWdlIjogZmxvYXQobnAubWVhbihbcm93WyJmaXhlZF9yaXNrX2NvdmVyZWQiXSBmb3Igcm93IGluIHN1',
    'YnNldF0pKSwKICAgICAgICAgICAgImZpeGVkX2dhcF9jb3ZlcmFnZSI6IGZsb2F0KG5wLm1lYW4oW3Jvd1siZml4ZWRfZ2Fw',
    'X2NvdmVyZWQiXSBmb3Igcm93IGluIHN1YnNldF0pKSwKICAgICAgICB9KQogICAgc3VtbWFyeSA9IHsKICAgICAgICAicHJv',
    'dG9jb2wiOiAiSUlELXdpdGgtcmVwbGFjZW1lbnQgZmVlZGJhY2sgZnJvbSBlYWNoIGZpeGVkIGNocm9ub2xvZ2ljYWwgMjU2',
    'LXJvdyBwb29sIiwKICAgICAgICAicG9wdWxhdGlvbl90YXJnZXQiOiAiZXhhY3Qgb2JqZWN0aXZlIG9mIHRoZSBvcmlnaW5h',
    'bCAyNTYtcm93IHBvb2wgYXQgZWFjaCBjdXJyZW50IGFjdGlvbiIsCiAgICAgICAgIkFQUDAxX2Fzc3VtcHRpb25fc2F0aXNm',
    'aWVkX2J5X2Rlc2lnbiI6IFRydWUsCiAgICAgICAgIm9ic2VydmF0aW9uYWxfVUNJX2NvdmVyYWdlX2NsYWltIjogRmFsc2Us',
    'CiAgICAgICAgImNvbmZpZGVuY2VfcGVyX3NlZWRfZm9yX2FsbF9yb3VuZHNfYW5kX2FjdGlvbnMiOiAxIC0gZGVsdGEsCiAg',
    'ICAgICAgInNlZWRzIjogbGlzdChzZWVkcyksICJob3Jpem9uIjogaG9yaXpvbiwgInNhbXBsZV9zaXplIjogbiwKICAgICAg',
    'ICAicmlza19ncmlkX3NwYWNpbmciOiBzcGFjaW5nLAogICAgICAgICJyaXNrX3JhZGl1cyI6IHJpc2tfcmFkaXVzLAogICAg',
    'ICAgICJtZWFuX2dhcF9yYWRpdXMiOiBmbG9hdChucC5tZWFuKFtyb3dbImdhcF9yYWRpdXMiXSBmb3Igcm93IGluIHJvd3Nd',
    'KSksCiAgICAgICAgIm1heF9yaXNrX2Vycm9yX2F0X3BsYXllZF9hY3Rpb25zIjogZmxvYXQobnAubWF4KFtyb3dbInJpc2tf',
    'ZXJyb3IiXSBmb3Igcm93IGluIHJvd3NdKSksCiAgICAgICAgIm1heF9nYXBfZXJyb3JfYXRfcGxheWVkX2FjdGlvbnMiOiBm',
    'bG9hdChucC5tYXgoW3Jvd1siZ2FwX2Vycm9yIl0gZm9yIHJvdyBpbiByb3dzXSkpLAogICAgICAgICJmcmFjdGlvbl9vZl9z',
    'ZWVkc19hbGxfcGxheWVkX2FjdGlvbnNfY292ZXJlZCI6IGZsb2F0KG5wLm1lYW4oW3JbImFsbF9wbGF5ZWRfYWN0aW9uc19j',
    'b3ZlcmVkIl0gZm9yIHIgaW4gcGVyX3NlZWRdKSksCiAgICAgICAgIm1lYW5fZml4ZWRfMDA1X3Jpc2tfY292ZXJhZ2UiOiBm',
    'bG9hdChucC5tZWFuKFtyWyJmaXhlZF9yaXNrX2NvdmVyYWdlIl0gZm9yIHIgaW4gcGVyX3NlZWRdKSksCiAgICAgICAgIm1l',
    'YW5fZml4ZWRfMDA1X2dhcF9jb3ZlcmFnZSI6IGZsb2F0KG5wLm1lYW4oW3JbImZpeGVkX2dhcF9jb3ZlcmFnZSJdIGZvciBy',
    'IGluIHBlcl9zZWVkXSkpLAogICAgICAgICJjb3ZlcmFnZV9mb3JfZXZlcnlfYWN0aW9uX2VtcGlyaWNhbGx5X2NoZWNrZWQi',
    'OiBGYWxzZSwKICAgICAgICAicGVyX3NlZWQiOiBwZXJfc2VlZCwKICAgIH0KICAgIChvdXRwdXQgLyAiY29udHJvbGxlZF9y',
    'ZXNhbXBsaW5nX3N1bW1hcnkuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhzdW1tYXJ5LCBpbmRlbnQ9MiksIGVuY29k',
    'aW5nPSJ1dGYtOCIpCiAgICByZXR1cm4gc3VtbWFyeQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJzZXIg',
    'PSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGEiLCB0eXBlPVBhdGgs',
    'IHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9',
    'VHJ1ZSkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBwcmludChqc29uLmR1bXBzKHJ1bihhcmdzLmRhdGEs',
    'IGFyZ3Mub3V0cHV0KSwgaW5kZW50PTIpKQo=',
]))
(CODE / "experiments" / "controlled_bank_resampling.py").write_bytes(CONTROLLED_SOURCE)
if RUN_OK:
    command = [sys.executable, str(CODE / "experiments" / "controlled_bank_resampling.py"),
               "--data", str(DATA), "--output", str(RESULTS / "controlled_resampling")]
    outcome = subprocess.run(command, cwd=CODE, text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(outcome.stdout[-9000:])
    if outcome.returncode == 0:
        CONTROLLED = json.loads((RESULTS / "controlled_resampling" /
                                 "controlled_resampling_summary.json").read_text(encoding="utf-8"))
        record("controlled_resampling", "passed", "10 seeds, 128 rounds each")
    else:
        RUN_OK = False
        record("controlled_resampling", "failed", f"exit={outcome.returncode}")
else:
    record("controlled_resampling", "skipped", "earlier gate failed")


## 6. Check the predictable-comparator certificate


In [ ]:
PREDICTABLE_SOURCE = base64.b64decode("".join([
    'IiIiQ29uZGl0aW9uYWwgRVhQLTEwIGNlcnRpZmljYXRlIGFnYWluc3QgdGhlIHByZWRlY2xhcmVkIG5ldXRyYWwgYWN0aW9u',
    'LgoKVGhlIGFjdGlvbi1zcGVjaWZpYyByYWRpaSBhcmUgdmFsaWQgb25seSBmb3IgcGFzdC1tZWFzdXJhYmxlIGFjdGlvbnMg',
    'YW5kCmNvbXBhcmF0b3JzIHVuZGVyIGNvbmRpdGlvbmFsIHdpdGhpbi1iYXRjaCBJSUQgc2FtcGxpbmcuIFRoZSBVQ0kgY2hy',
    'b25vbG9neQpkb2VzIG5vdCBlc3RhYmxpc2ggdGhhdCBzYW1wbGluZyBjb25kaXRpb24uCiIiIgoKaW1wb3J0IGFyZ3BhcnNl',
    'CmltcG9ydCBjc3YKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzeXMK',
    'CmltcG9ydCBudW1weSBhcyBucAoKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFy',
    'ZW50c1sxXSAvICJzcmMiKSkKZnJvbSBpbnRlcnZhbF9wYXJldG8uYWxnb3JpdGhtcyBpbXBvcnQgTWlkcG9pbnRPR0QKZnJv',
    'bSBpbnRlcnZhbF9wYXJldG8uYXBwbGljYXRpb25zLnN0cmVhbWluZyBpbXBvcnQgQ2xhc3NpZmljYXRpb25CYXRjaCwgb2Jq',
    'ZWN0aXZlX2FuZF9zdWJncmFkaWVudHMKCgpkZWYgYWN0aW9uX3JhZGlpKGFjdGlvbjogbnAubmRhcnJheSwgbjA6IGludCwg',
    'bjE6IGludCwgaG9yaXpvbjogaW50LAogICAgICAgICAgICAgICAgIGRlbHRhOiBmbG9hdCA9IDAuMDUpIC0+IHR1cGxlW2Zs',
    'b2F0LCBmbG9hdF06CiAgICAiIiJUd28tc2lkZWQgcmlzayBhbmQgZ3JvdXAtZ2FwIHJhZGlpIGZvciBhIHByZWRpY3RhYmxl',
    'IGFjdGlvbi4iIiIKICAgIGlmIGFjdGlvbi5zaGFwZSAhPSAoMjQsKSBvciBucC5hbnkoYWN0aW9uIDwgMCkgb3IgbnAuYW55',
    'KGFjdGlvbiA+IDEpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImV4cGVjdGVkIGEgMjQtZGltZW5zaW9uYWwgdW5pdC1i',
    'b3ggYWN0aW9uIikKICAgIGlmIG1pbihuMCwgbjEsIGhvcml6b24pIDwgMSBvciBub3QgMCA8IGRlbHRhIDwgMToKICAgICAg',
    'ICByYWlzZSBWYWx1ZUVycm9yKCJpbnZhbGlkIGJhdGNoIGNvdW50cyBvciBjb25maWRlbmNlIikKICAgIHRoZXRhID0gYWN0',
    'aW9uWzo3XSAtIGFjdGlvbls3OjE0XQogICAgaW50ZXJjZXB0ID0gZmxvYXQobnAuc3VtKGFjdGlvblsxNDoxOV0gLSBhY3Rp',
    'b25bMTk6MjRdKSkKICAgIGFtcGxpdHVkZSA9IGZsb2F0KG5wLnN1bShucC5hYnModGhldGEpKSkKICAgIHNjb3JlX2JvdW5k',
    'ID0gYW1wbGl0dWRlICsgYWJzKGludGVyY2VwdCkKICAgIHJpc2sgPSBzY29yZV9ib3VuZCAqIG1hdGguc3FydChtYXRoLmxv',
    'Zyg0ICogaG9yaXpvbiAvIGRlbHRhKSAvICgyICogKG4wICsgbjEpKSkKICAgIGdyb3VwX2ZhY3RvciA9IG1hdGguc3FydCgy',
    'ICogbWF0aC5sb2coOCAqIGhvcml6b24gLyBkZWx0YSkpCiAgICBnYXAgPSBhbXBsaXR1ZGUgKiBncm91cF9mYWN0b3IgKiAo',
    'MSAvIG1hdGguc3FydChuMCkgKyAxIC8gbWF0aC5zcXJ0KG4xKSkKICAgIHJldHVybiByaXNrLCBnYXAKCgpkZWYgcnVuKGRh',
    'dGFfcGF0aDogUGF0aCwgb3V0cHV0OiBQYXRoLCBldGE6IGZsb2F0ID0gMC4wNSwKICAgICAgICBkZWx0YTogZmxvYXQgPSAw',
    'LjA1KSAtPiBkaWN0OgogICAgd2l0aCBucC5sb2FkKGRhdGFfcGF0aCwgYWxsb3dfcGlja2xlPUZhbHNlKSBhcyBkYXRhOgog',
    'ICAgICAgIHggPSBucC5hc2FycmF5KGRhdGFbImZlYXR1cmVzIl0sIGZsb2F0KQogICAgICAgIHkgPSBucC5hc2FycmF5KGRh',
    'dGFbImxhYmVscyJdLCBmbG9hdCkKICAgICAgICBnID0gbnAuYXNhcnJheShkYXRhWyJncm91cHMiXSwgaW50KQogICAgICAg',
    'IHdlaWdodHMgPSBucC5hc2FycmF5KGRhdGFbIndlaWdodHMiXSwgZmxvYXQpCiAgICBpZiB4Lm5kaW0gIT0gMyBvciB4LnNo',
    'YXBlWzJdICE9IDI0IG9yIHkuc2hhcGUgIT0geC5zaGFwZVs6Ml0gb3IgZy5zaGFwZSAhPSB5LnNoYXBlOgogICAgICAgIHJh',
    'aXNlIFZhbHVlRXJyb3IoIkVYUC0xMCBhcnJheXMgcmVxdWlyZWQiKQogICAgaG9yaXpvbiwgbiwgZGltZW5zaW9uID0geC5z',
    'aGFwZQogICAgY29tcGFyYXRvciA9IG5wLmZ1bGwoZGltZW5zaW9uLCAwLjUpCiAgICBsZWFybmVyID0gTWlkcG9pbnRPR0Qo',
    'ZGltZW5zaW9uLCBldGEpCiAgICByb3dzID0gW10KICAgIGdyYWRpZW50X3NxdWFyZV9zdW0gPSAwLjAKICAgIGZvciB0IGlu',
    'IHJhbmdlKGhvcml6b24pOgogICAgICAgIGFjdGlvbiA9IGxlYXJuZXIuYWN0KCkKICAgICAgICBiYXRjaCA9IENsYXNzaWZp',
    'Y2F0aW9uQmF0Y2goeFt0XSwgeVt0XSwgZ1t0XSkKICAgICAgICBvYnNlcnZlZCwgZ3JhZGllbnRzID0gb2JqZWN0aXZlX2Fu',
    'ZF9zdWJncmFkaWVudHMoYmF0Y2gsIGFjdGlvbikKICAgICAgICBjb21wYXJhdG9yX29ic2VydmVkLCBfID0gb2JqZWN0aXZl',
    'X2FuZF9zdWJncmFkaWVudHMoYmF0Y2gsIGNvbXBhcmF0b3IpCiAgICAgICAgbjAgPSBpbnQobnAuc3VtKGdbdF0gPT0gMCkp',
    'CiAgICAgICAgbjEgPSBuIC0gbjAKICAgICAgICByaXNrX3JhZGl1cywgZ2FwX3JhZGl1cyA9IGFjdGlvbl9yYWRpaShhY3Rp',
    'b24sIG4wLCBuMSwgaG9yaXpvbiwgZGVsdGEpCiAgICAgICAgZ3JhZGllbnQgPSB3ZWlnaHRzW3RdIEAgZ3JhZGllbnRzCiAg',
    'ICAgICAgZ3JhZGllbnRfc3F1YXJlX3N1bSArPSBmbG9hdChncmFkaWVudCBAIGdyYWRpZW50KQogICAgICAgIHJvd3MuYXBw',
    'ZW5kKHsKICAgICAgICAgICAgImJhdGNoIjogdCArIDEsICJuMCI6IG4wLCAibjEiOiBuMSwKICAgICAgICAgICAgIm9ic2Vy',
    'dmVkX3dlaWdodGVkX2FjdGlvbiI6IGZsb2F0KHdlaWdodHNbdF0gQCBvYnNlcnZlZCksCiAgICAgICAgICAgICJvYnNlcnZl',
    'ZF93ZWlnaHRlZF9jb21wYXJhdG9yIjogZmxvYXQod2VpZ2h0c1t0XSBAIGNvbXBhcmF0b3Jfb2JzZXJ2ZWQpLAogICAgICAg',
    'ICAgICAicmlza19yYWRpdXNfYXRfYWN0aW9uIjogcmlza19yYWRpdXMsCiAgICAgICAgICAgICJnYXBfcmFkaXVzX2F0X2Fj',
    'dGlvbiI6IGdhcF9yYWRpdXMsCiAgICAgICAgICAgICJ3ZWlnaHRlZF90cmFuc2Zlcl9yYWRpdXMiOiBmbG9hdCh3ZWlnaHRz',
    'W3QsIDBdICogcmlza19yYWRpdXMgKyB3ZWlnaHRzW3QsIDFdICogZ2FwX3JhZGl1cyksCiAgICAgICAgICAgICJncmFkaWVu',
    'dF9zcXVhcmVkX25vcm0iOiBmbG9hdChncmFkaWVudCBAIGdyYWRpZW50KSwKICAgICAgICB9KQogICAgICAgIGxlYXJuZXIu',
    'b2JzZXJ2ZV9ncmFkaWVudChncmFkaWVudCkKICAgIG91dHB1dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUp',
    'CiAgICB3aXRoIChvdXRwdXQgLyAicHJlZGljdGFibGVfY29tcGFyYXRvcl9yb3VuZHMuY3N2Iikub3BlbigidyIsIG5ld2xp',
    'bmU9IiIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGhhbmRsZToKICAgICAgICB3cml0ZXIgPSBjc3YuRGljdFdyaXRlcihoYW5k',
    'bGUsIGZpZWxkbmFtZXM9bGlzdChyb3dzWzBdKSkKICAgICAgICB3cml0ZXIud3JpdGVoZWFkZXIoKQogICAgICAgIHdyaXRl',
    'ci53cml0ZXJvd3Mocm93cykKICAgIHRyYW5zZmVyID0gc3VtKHJvd1sid2VpZ2h0ZWRfdHJhbnNmZXJfcmFkaXVzIl0gZm9y',
    'IHJvdyBpbiByb3dzKQogICAgb3B0aW1pemF0aW9uID0gZXRhICogZ3JhZGllbnRfc3F1YXJlX3N1bSAvIDIgICMgY29tcGFy',
    'YXRvciBlcXVhbHMgaW5pdGlhbCBhY3Rpb24KICAgIHN1bW1hcnkgPSB7CiAgICAgICAgInN0YXR1cyI6ICJjb25kaXRpb25h',
    'bF9wcmVkaWN0YWJsZV9jb21wYXJhdG9yX2NlcnRpZmljYXRlIiwKICAgICAgICAic2FtcGxpbmdfYXNzdW1wdGlvbiI6ICJj',
    'dXJyZW50IGJhdGNoIElJRCBjb25kaXRpb25hbCBvbiBwYXN0OyB1bnZlcmlmaWVkIGZvciBvYnNlcnZhdGlvbmFsIFVDSSBj',
    'aHJvbm9sb2d5IiwKICAgICAgICAiY29tcGFyYXRvciI6ICJwcmVkZWNsYXJlZCBuZXV0cmFsIGFjdGlvbiB3PSgwLjUsLi4u',
    'LDAuNSkiLAogICAgICAgICJjb21wYXJhdG9yX3NlbGVjdGVkX3VzaW5nX2N1cnJlbnRfb3JfZnV0dXJlX2RhdGEiOiBGYWxz',
    'ZSwKICAgICAgICAiY29uZmlkZW5jZV9mb3JfYWxsX3JvdW5kc19hdF9wbGF5ZWRfYW5kX2NvbXBhcmF0b3JfYWN0aW9ucyI6',
    'IDEgLSBkZWx0YSwKICAgICAgICAiaG9yaXpvbiI6IGhvcml6b24sCiAgICAgICAgIm1lYW5fcmlza19yYWRpdXNfYXRfcGxh',
    'eWVkX2FjdGlvbiI6IGZsb2F0KG5wLm1lYW4oW3JbInJpc2tfcmFkaXVzX2F0X2FjdGlvbiJdIGZvciByIGluIHJvd3NdKSks',
    'CiAgICAgICAgIm1lYW5fZ2FwX3JhZGl1c19hdF9wbGF5ZWRfYWN0aW9uIjogZmxvYXQobnAubWVhbihbclsiZ2FwX3JhZGl1',
    'c19hdF9hY3Rpb24iXSBmb3IgciBpbiByb3dzXSkpLAogICAgICAgICJUSE0wMl9RX1RfYm91bmQiOiB0cmFuc2ZlciwKICAg',
    'ICAgICAicmVhbGl6ZWRfZ3JhZGllbnRfb3B0aW1pemF0aW9uX2JvdW5kIjogb3B0aW1pemF0aW9uLAogICAgICAgICJjb25k',
    'aXRpb25hbF9sYXRlbnRfcmVncmV0X3VwcGVyX2JvdW5kIjogb3B0aW1pemF0aW9uICsgdHJhbnNmZXIsCiAgICAgICAgIm9i',
    'c2VydmVkX2N1bXVsYXRpdmVfYWN0aW9uX21pbnVzX2NvbXBhcmF0b3IiOiBzdW0oCiAgICAgICAgICAgIHJbIm9ic2VydmVk',
    'X3dlaWdodGVkX2FjdGlvbiJdIC0gclsib2JzZXJ2ZWRfd2VpZ2h0ZWRfY29tcGFyYXRvciJdIGZvciByIGluIHJvd3MpLAog',
    'ICAgICAgICJvYnNlcnZhdGlvbmFsX3BvcHVsYXRpb25fY292ZXJhZ2VfY2xhaW0iOiBGYWxzZSwKICAgIH0KICAgIChvdXRw',
    'dXQgLyAicHJlZGljdGFibGVfY29tcGFyYXRvcl9zdW1tYXJ5Lmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVt',
    'cHMoc3VtbWFyeSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgcmV0dXJuIHN1bW1hcnkKCgppZiBfX25hbWVf',
    'XyA9PSAiX19tYWluX18iOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcGFyc2VyLmFkZF9h',
    'cmd1bWVudCgiLS1kYXRhIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1v',
    'dXRwdXQiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcHJp',
    'bnQoanNvbi5kdW1wcyhydW4oYXJncy5kYXRhLCBhcmdzLm91dHB1dCksIGluZGVudD0yKSkK',
]))
CHECK_SOURCE = base64.b64decode("".join([
    'IiIiQ2hlY2sgdGhlIHByZWRpY3RhYmxlLWNvbXBhcmF0b3IgY2VydGlmaWNhdGUgd2l0aCBrbm93biBmaW5pdGUgcG9vbHMu',
    'IiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzeXMKCmlt',
    'cG9ydCBudW1weSBhcyBucAoKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50',
    'c1sxXSAvICJzcmMiKSkKZnJvbSBpbnRlcnZhbF9wYXJldG8uYWxnb3JpdGhtcyBpbXBvcnQgTWlkcG9pbnRPR0QKZnJvbSBp',
    'bnRlcnZhbF9wYXJldG8uYXBwbGljYXRpb25zLnN0cmVhbWluZyBpbXBvcnQgQ2xhc3NpZmljYXRpb25CYXRjaCwgb2JqZWN0',
    'aXZlX2FuZF9zdWJncmFkaWVudHMKZnJvbSBwcmVkaWN0YWJsZV9jb21wYXJhdG9yX2NlcnRpZmljYXRlIGltcG9ydCBhY3Rp',
    'b25fcmFkaWkKCgpkZWYgcnVuKGRhdGFfcGF0aDogUGF0aCwgb3V0cHV0OiBQYXRoLCBzZWVkczogdHVwbGVbaW50LCAuLi5d',
    'ID0gdHVwbGUocmFuZ2UoMTApKSwKICAgICAgICBldGE6IGZsb2F0ID0gMC4wNSwgZGVsdGE6IGZsb2F0ID0gMC4wNSkgLT4g',
    'ZGljdDoKICAgIHdpdGggbnAubG9hZChkYXRhX3BhdGgsIGFsbG93X3BpY2tsZT1GYWxzZSkgYXMgZGF0YToKICAgICAgICB4',
    'ID0gbnAuYXNhcnJheShkYXRhWyJmZWF0dXJlcyJdLCBmbG9hdCkKICAgICAgICB5ID0gbnAuYXNhcnJheShkYXRhWyJsYWJl',
    'bHMiXSwgZmxvYXQpCiAgICAgICAgZyA9IG5wLmFzYXJyYXkoZGF0YVsiZ3JvdXBzIl0sIGludCkKICAgICAgICB3ZWlnaHRz',
    'ID0gbnAuYXNhcnJheShkYXRhWyJ3ZWlnaHRzIl0sIGZsb2F0KQogICAgaG9yaXpvbiwgbiwgZGltZW5zaW9uID0geC5zaGFw',
    'ZQogICAgY29tcGFyYXRvciA9IG5wLmZ1bGwoZGltZW5zaW9uLCAwLjUpCiAgICByZXN1bHRzID0gW10KICAgIGZvciBzZWVk',
    'IGluIHNlZWRzOgogICAgICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKQogICAgICAgIGxlYXJuZXIgPSBN',
    'aWRwb2ludE9HRChkaW1lbnNpb24sIGV0YSkKICAgICAgICByaXNrX2hpdHMgPSBbXQogICAgICAgIGdhcF9oaXRzID0gW10K',
    'ICAgICAgICB0cmFuc2ZlciA9IDAuMAogICAgICAgIG9wdGltaXphdGlvbiA9IDAuMAogICAgICAgIGxhdGVudF9yZWdyZXQg',
    'PSAwLjAKICAgICAgICBmb3IgdCBpbiByYW5nZShob3Jpem9uKToKICAgICAgICAgICAgYWN0aW9uID0gbGVhcm5lci5hY3Qo',
    'KQogICAgICAgICAgICBkcmF3ID0gcm5nLmludGVnZXJzKDAsIG4sIHNpemU9bikKICAgICAgICAgICAgc2FtcGxlID0gQ2xh',
    'c3NpZmljYXRpb25CYXRjaCh4W3QsIGRyYXddLCB5W3QsIGRyYXddLCBnW3QsIGRyYXddKQogICAgICAgICAgICBwb29sID0g',
    'Q2xhc3NpZmljYXRpb25CYXRjaCh4W3RdLCB5W3RdLCBnW3RdKQogICAgICAgICAgICBuMCA9IGludChucC5zdW0oc2FtcGxl',
    'Lmdyb3VwcyA9PSAwKSkKICAgICAgICAgICAgbjEgPSBuIC0gbjAKICAgICAgICAgICAgaWYgbWluKG4wLCBuMSkgPCAxOgog',
    'ICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJlbXB0eSBzYW1wbGVkIGFnZSBncm91cCIpCiAgICAgICAgICAg',
    'IG9ic2VydmVkLCBncmFkaWVudHMgPSBvYmplY3RpdmVfYW5kX3N1YmdyYWRpZW50cyhzYW1wbGUsIGFjdGlvbikKICAgICAg',
    'ICAgICAgcG9wdWxhdGlvbiwgXyA9IG9iamVjdGl2ZV9hbmRfc3ViZ3JhZGllbnRzKHBvb2wsIGFjdGlvbikKICAgICAgICAg',
    'ICAgbmV1dHJhbCwgXyA9IG9iamVjdGl2ZV9hbmRfc3ViZ3JhZGllbnRzKHBvb2wsIGNvbXBhcmF0b3IpCiAgICAgICAgICAg',
    'IHJyLCBnciA9IGFjdGlvbl9yYWRpaShhY3Rpb24sIG4wLCBuMSwgaG9yaXpvbiwgZGVsdGEpCiAgICAgICAgICAgIHJpc2tf',
    'aGl0cy5hcHBlbmQoYWJzKG9ic2VydmVkWzBdIC0gcG9wdWxhdGlvblswXSkgPD0gcnIgKyAxZS0xMikKICAgICAgICAgICAg',
    'Z2FwX2hpdHMuYXBwZW5kKGFicyhvYnNlcnZlZFsxXSAtIHBvcHVsYXRpb25bMV0pIDw9IGdyICsgMWUtMTIpCiAgICAgICAg',
    'ICAgIHRyYW5zZmVyICs9IHdlaWdodHNbdCwgMF0gKiByciArIHdlaWdodHNbdCwgMV0gKiBncgogICAgICAgICAgICBncmFk',
    'aWVudCA9IHdlaWdodHNbdF0gQCBncmFkaWVudHMKICAgICAgICAgICAgb3B0aW1pemF0aW9uICs9IGV0YSAqIGZsb2F0KGdy',
    'YWRpZW50IEAgZ3JhZGllbnQpIC8gMgogICAgICAgICAgICBsYXRlbnRfcmVncmV0ICs9IGZsb2F0KHdlaWdodHNbdF0gQCAo',
    'cG9wdWxhdGlvbiAtIG5ldXRyYWwpKQogICAgICAgICAgICBsZWFybmVyLm9ic2VydmVfZ3JhZGllbnQoZ3JhZGllbnQpCiAg',
    'ICAgICAgcmVzdWx0cy5hcHBlbmQoewogICAgICAgICAgICAic2VlZCI6IHNlZWQsCiAgICAgICAgICAgICJyaXNrX2NvdmVy',
    'ZWQiOiBpbnQoc3VtKHJpc2tfaGl0cykpLAogICAgICAgICAgICAiZ2FwX2NvdmVyZWQiOiBpbnQoc3VtKGdhcF9oaXRzKSks',
    'CiAgICAgICAgICAgICJqb2ludF9hbGxfcm91bmRzX2NvdmVyZWQiOiBib29sKGFsbChyaXNrX2hpdHMpIGFuZCBhbGwoZ2Fw',
    'X2hpdHMpKSwKICAgICAgICAgICAgInBvcHVsYXRpb25fcmVncmV0X3ZzX25ldXRyYWwiOiBsYXRlbnRfcmVncmV0LAogICAg',
    'ICAgICAgICAiY29uZGl0aW9uYWxfYm91bmQiOiBvcHRpbWl6YXRpb24gKyB0cmFuc2ZlciwKICAgICAgICAgICAgImJvdW5k',
    'X2hvbGRzX2Zvcl9yZWFsaXplZF9wb29sIjogYm9vbChsYXRlbnRfcmVncmV0IDw9IG9wdGltaXphdGlvbiArIHRyYW5zZmVy',
    'ICsgMWUtMTApLAogICAgICAgIH0pCiAgICBzdW1tYXJ5ID0gewogICAgICAgICJwcm90b2NvbCI6ICJJSUQtd2l0aC1yZXBs',
    'YWNlbWVudCBmZWVkYmFjayBmcm9tIGVhY2ggZml4ZWQgY2hyb25vbG9naWNhbCBwb29sIiwKICAgICAgICAiQVBQMDFfc2Ft',
    'cGxpbmdfYXNzdW1wdGlvbl9zYXRpc2ZpZWRfYnlfZGVzaWduIjogVHJ1ZSwKICAgICAgICAiY29uZmlkZW5jZV9wZXJfc2Vl',
    'ZF9zaW11bHRhbmVvdXNseV9vdmVyX3JvdW5kc19hdF9wbGF5ZWRfYWN0aW9ucyI6IDEgLSBkZWx0YSwKICAgICAgICAic2Vl',
    'ZHMiOiBsaXN0KHNlZWRzKSwgImhvcml6b24iOiBob3Jpem9uLAogICAgICAgICJhbGxfc2VlZHNfYWxsX3BsYXllZF9hY3Rp',
    'b25zX2NvdmVyZWQiOiBhbGwoclsiam9pbnRfYWxsX3JvdW5kc19jb3ZlcmVkIl0gZm9yIHIgaW4gcmVzdWx0cyksCiAgICAg',
    'ICAgImFsbF9yZWFsaXplZF9wb29sX3JlZ3JldF9ib3VuZHNfaG9sZCI6IGFsbChyWyJib3VuZF9ob2xkc19mb3JfcmVhbGl6',
    'ZWRfcG9vbCJdIGZvciByIGluIHJlc3VsdHMpLAogICAgICAgICJwZXJfc2VlZCI6IHJlc3VsdHMsCiAgICB9CiAgICBvdXRw',
    'dXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgKG91dHB1dCAvICJwcmVkaWN0YWJsZV9jb250cm9s',
    'bGVkX3N1bW1hcnkuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhzdW1tYXJ5LCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1',
    'dGYtOCIpCiAgICByZXR1cm4gc3VtbWFyeQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJzZXIgPSBhcmdw',
    'YXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGEiLCB0eXBlPVBhdGgsIHJlcXVp',
    'cmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkK',
    'ICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBwcmludChqc29uLmR1bXBzKHJ1bihhcmdzLmRhdGEsIGFyZ3Mu',
    'b3V0cHV0KSwgaW5kZW50PTIpKQo=',
]))
PREDICTABLE_PROOF = base64.b64decode("".join([
    'IyBFWFAtMTAgaW50ZXJ2YWxzIGZvciBhIHByZWRpY3RhYmxlIGNvbXBhcmF0b3IKCioqU3RhdHVzOiBjb25kaXRpb25hbCBw',
    'cm9wb3NpdGlvbiwgcHJvdmVkLioqIFRoaXMgcmVmaW5lcyB0aGUgdGFyZ2V0IG9mCltBUFAtMDFdKHN0cmVhbWluZ19wb3B1',
    'bGF0aW9uX2ludGVydmFscy5tZCkuIEl0IGFwcGxpZXMgdG8gYSBwcmVkZWNsYXJlZCBvcgpwYXN0LW1lYXN1cmFibGUgY29t',
    'cGFyYXRvciBwYXRoLCBub3QgYSBoaW5kc2lnaHQtc2VsZWN0ZWQgY29tcGFyYXRvciBvciBldmVyeQpib3ggYWN0aW9uLiBU',
    'aGUgb3JpZ2luYWwgVUNJIGNocm9ub2xvZ3kgZG9lcyBub3QgdmVyaWZ5IHdpdGhpbi1iYXRjaCBJSUQuCgojIyBQcm9wb3Np',
    'dGlvbgoKQXNzdW1lIEFQUC0wMSdzIGNvbmRpdGlvbmFsIElJRCBhbmQgYm91bmRlZC1mZWF0dXJlIG1vZGVsLCBmaXhlZCBi',
    'YXRjaCBzaXplCmBuYCwgaG9yaXpvbiBgVGAsIGFuZCBwb3N0LWRlY2lzaW9uIHNhbXBsaW5nLiBMZXQgdGhlIGxlYXJuZXIg',
    'YWN0aW9uIGB3X3RgCmFuZCBjb21wYXJhdG9yIGB1X3RgIGJvdGggYmUgbWVhc3VyYWJsZSBmcm9tIHRoZSBwYXN0IGJlZm9y',
    'ZSBiYXRjaCBgdGAuCkZvciBFWFAtMTAsIHVzZSB0aGUgbmV1dHJhbCBmaXhlZCBjb21wYXJhdG9yIGB1X3Q9KDEvMiwuLi4s',
    'MS8yKWAuIEl0IGluZHVjZXMKemVybyBzY29yZSwgc28gaXRzIHJpc2sgaXMgZXhhY3RseSBgbG9nIDJgLCBpdHMgZ3JvdXAg',
    'Z2FwIGV4YWN0bHkgemVybywgYW5kCml0cyByZXNvdXJjZSBjb3N0IGV4YWN0bHkgdGhyZWUgZm9yIGJvdGggc2FtcGxlIGFu',
    'ZCBwb3B1bGF0aW9uLgoKRm9yIHRoZSBwbGF5ZWQgYWN0aW9uLCB3cml0ZSBgzrhfdD13X3RbMDo3XS13X3RbNzoxNF1gLApg',
    'Yl90Pc6jX3trPTB9XjQod190WzE0K2tdLXdfdFsxOStrXSlgLApgQV90PXx8zrhfdHx8XzFgLCBhbmQgYEJfdD1BX3QrfGJf',
    'dHxgLiBJZiB0aGUgY3VycmVudCBzYW1wbGUgaGFzIHBvc2l0aXZlCmdyb3VwIGNvdW50cyBgbl90MCxuX3QxYCwgc2V0Cgpg',
    'cl90LHJpc2s9Ql90IHNxcnQobG9nKDRUL860KS8oMm4pKWAsCgpgcl90LGdhcD1BX3Qgc3FydCgyIGxvZyg4VC/OtCkpCiAg',
    'ICAgICAgICAgICAoMS9zcXJ0KG5fdDApKzEvc3FydChuX3QxKSlgLAoKYW5kIGByX3QscmVzb3VyY2U9MGAuIFVzZSBhIHRy',
    'aXZpYWwgcmFuZ2UtYmFzZWQgZmFsbGJhY2sgaWYgZWl0aGVyIGdyb3VwCmlzIGFic2VudC4gVGhlbiwgd2l0aCBwcm9iYWJp',
    'bGl0eSBhdCBsZWFzdCBgMS3OtGAsIGFsbCBgVGAgcG9wdWxhdGlvbgpvYmplY3RpdmUgdmFsdWVzIGF0IHRoZSBwbGF5ZWQg',
    'YWN0aW9ucyBsaWUgaW4gdGhlIGNvcnJlc3BvbmRpbmcgZW1waXJpY2FsCmludGVydmFscyBzaW11bHRhbmVvdXNseS4gVGhl',
    'IGNvbXBhcmF0b3IgdmFsdWVzIGFyZSBleGFjdC4gVGhlIGV2ZW50IGlzCmNvbmRpdGlvbmFsIG9uIHRoZSBtb2RlbCBhc3N1',
    'bXB0aW9ucyBhbmQgYWxsb3dzIGFyYml0cmFyeSBkcmlmdCBhY3Jvc3MKYmF0Y2hlcy4KCkZvciB0aGUgRVhQLTEwIHdlaWdo',
    'dHMgYM67X3Q9KDAuOCwwLjE1LDAuMDUpYCBhbmQgbWlkcG9pbnQgT0dEIGdyYWRpZW50CmBzX3RgLCBwcm9qZWN0aW9uIGdp',
    'dmVzIHRoZSBmaXhlZC1jb21wYXJhdG9yIHBvcHVsYXRpb24tcmVncmV0IGJvdW5kCgpgzqNfdCDOu190wrdbaF90KHdfdCkt',
    'aF90KHUpXQog4omkIHx8d18xLXV8fMKyLygyzrcpKyjOty8yKc6jX3R8fHNfdHx8wrIKICAgK86jX3RbMC44IHJfdCxyaXNr',
    'KzAuMTUgcl90LGdhcF1gLgoKQmVjYXVzZSBgd18xPXVgLCB0aGUgaW5pdGlhbC1kaXN0YW5jZSB0ZXJtIGlzIHplcm8uIFRo',
    'aXMgYm91bmQgaXMKY29tcHV0YWJsZSBhZnRlciBmZWVkYmFjaywgYW5kIGl0cyBwcm9iYWJpbGl0eSBzdGF0ZW1lbnQgaXMg',
    'dmFsaWQgZXZlbgp0aG91Z2ggdGhlIGFjdGlvbnMgYWRhcHQgdG8gZWFybGllciBiYXRjaGVzLgoKIyMgUHJvb2YKCkNvbmRp',
    'dGlvbmFsIG9uIHRoZSBwYXN0LCBgd190YCBhbmQgYCjOuF90LGJfdClgIGFyZSBmaXhlZC4gVGhlIHNjb3JlIGlzIGluCmBb',
    'LUJfdCxCX3RdYC4gRm9yIGVpdGhlciBsYWJlbCwgbG9naXN0aWMgbG9zcyB0aGVyZWZvcmUgaGFzIHJhbmdlIGxlbmd0aApg',
    'Ql90YCwgc2luY2UKYGxvZygxK2V4cChCX3QpKS1sb2coMStleHAoLUJfdCkpPUJfdGAuIFR3by1zaWRlZCBIb2VmZmRpbmcK',
    'd2l0aCBgzrQvKDJUKWAgYWxsb2NhdGVkIHRvIGVhY2ggcm91bmQncyByaXNrIGdpdmVzIHRoZSBzdGF0ZWQgcmlzayByYWRp',
    'dXMuCgpXaXRoaW4gYSBncm91cCwgYM64X3TCt1NgIGxpZXMgaW4gYFstQV90LEFfdF1gLiBDb25kaXRpb25hbCBvbiBwb3Np',
    'dGl2ZSBncm91cApjb3VudHMsIHRoZSBvYnNlcnZhdGlvbnMgaW4gZ3JvdXAgYGdgIGFyZSBJSUQgZnJvbSBgUF90KFN8Rz1n',
    'KWAuClR3by1zaWRlZCBIb2VmZmRpbmcgd2l0aCBgzrQvKDRUKWAgYWxsb2NhdGVkIHRvIGVhY2ggZ3JvdXAgYW5kIHJvdW5k',
    'IGdpdmVzCmBBX3Qgc3FydCgyIGxvZyg4VC/OtCkvbl90ZylgIGZvciBlYWNoIGdyb3VwIG1lYW4uIFRoZSBhYnNvbHV0ZS12',
    'YWx1ZSBtYXAKaXMgMS1MaXBzY2hpdHosIHNvIHRoZSBzdW0gb2YgdGhlIHR3byBncm91cCBtZWFuIGVycm9ycyBib3VuZHMg',
    'dGhlIGdhcAplcnJvci4gVGhlIG5ldXRyYWwgY29tcGFyYXRvciBpcyBleGFjdCwgYW5kIHJlc291cmNlIGlzIGRldGVybWlu',
    'aXN0aWMuClVuaW9uaW5nIG92ZXIgcmlza3MsIGdyb3VwcywgYW5kIHJvdW5kcyBjb3N0cyBhdCBtb3N0IGDOtGAuCgpUaGUg',
    'cHJvamVjdGVkIE9HRCBpbmVxdWFsaXR5IGZvciB0aGUgZW1waXJpY2FsIGNvbnZleCBtaWRwb2ludCBzY2FsYXIKbG9zcyB0',
    'ZWxlc2NvcGVzIGFnYWluc3QgdGhlIGZpeGVkIGNvbXBhcmF0b3IgdG8KYM6jX3RbZ190KHdfdCktZ190KHUpXeKJpHx8d18x',
    'LXV8fMKyLygyzrcpKyjOty8yKc6jX3R8fHNfdHx8wrJgLgpBdCBlYWNoIHBsYXllZCBhY3Rpb24sIHBvcHVsYXRpb24gbWlu',
    'dXMgZW1waXJpY2FsIHNjYWxhciBsb3NzIGlzIGF0IG1vc3QKdGhlIHdlaWdodGVkIHN1bSBvZiB0aGUgdHdvIHJhZGlpLiBB',
    'dCB0aGUgbmV1dHJhbCBjb21wYXJhdG9yIHRoZQpwb3B1bGF0aW9uIGFuZCBlbXBpcmljYWwgbG9zc2VzIGNvaW5jaWRlLiBB',
    'ZGQgdGhlc2UgaW5lcXVhbGl0aWVzLiDilqEKCiMjIEludGVycHJldGF0aW9uCgpUaGlzIHJlc3VsdCBtYWtlcyBhIHZhbGlk',
    'IGJ1dCBuYXJyb3dlciBjb21wYXJpc29uIHRoYW4gVEhNLTAyJ3MKaGluZHNpZ2h0LWNvbXBhcmF0b3IgZ3VhcmFudGVlLiBB',
    'IGNvbXBhcmF0b3IgY2hvc2VuIGFmdGVyIHNlZWluZyB0aGUKY3VycmVudCBvciBmdXR1cmUgZGF0YSBuZWVkIG5vdCBzYXRp',
    'c2Z5IHRoaXMgZXZlbnQuIFRoZSBvcmlnaW5hbCBVQ0kKYXBwbGljYXRpb24gZ2l2ZXMgb25seSBhIGNvbmRpdGlvbmFsIG51',
    'bWVyaWMgY2VydGlmaWNhdGUsIGJlY2F1c2UgaXRzCnB1Ymxpc2hlZCByb3cgb3JkZXIgZG9lcyBub3QgZXN0YWJsaXNoIElJ',
    'RCBzYW1wbGluZyB3aXRoaW4gZWFjaCBiYXRjaC4KVGhlIGNvbnRyb2xsZWQgd2l0aC1yZXBsYWNlbWVudCByZXBsYXkgZG9l',
    'cyBzYXRpc2Z5IHRoYXQgYXNzdW1wdGlvbiBieQpkZXNpZ24uIFRoZSBhZ2UtZ3JvdXAgc2NvcmUgZ2FwIGlzIGEgc3Vycm9n',
    'YXRlIG1lYXN1cmU7IHRoaXMgcmVzdWx0IGRvZXMKbm90IGltcGx5IG91dGNvbWUgZmFpcm5lc3MuCg==',
]))
(CODE / "experiments" / "predictable_comparator_certificate.py").write_bytes(PREDICTABLE_SOURCE)
(CODE / "experiments" / "check_predictable_controlled.py").write_bytes(CHECK_SOURCE)
(RESULTS / "predictable_comparator_intervals.md").write_bytes(PREDICTABLE_PROOF)
if RUN_OK:
    for stage, script, destination in [
        ("predictable_certificate", "predictable_comparator_certificate.py", "predictable_comparator"),
        ("predictable_controlled", "check_predictable_controlled.py", "predictable_controlled"),
    ]:
        command = [sys.executable, str(CODE / "experiments" / script),
                   "--data", str(DATA), "--output", str(RESULTS / destination)]
        outcome = subprocess.run(command, cwd=CODE, text=True,
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(outcome.stdout[-7000:])
        if outcome.returncode == 0:
            record(stage, "passed", destination)
        else:
            RUN_OK = False
            record(stage, "failed", f"exit={outcome.returncode}")
            break
else:
    record("predictable_certificate", "skipped", "earlier gate failed")
    record("predictable_controlled", "skipped", "earlier gate failed")


## 7. Zip all EXP-10 outputs


In [ ]:
import zipfile
(RESULTS / "notebook_stages.json").write_text(
    json.dumps({"run_ok": RUN_OK, "stages": STAGES}, indent=2), encoding="utf-8")
if BOOTSTRAP_ERROR is None:
    shutil.copy2(SOURCE_META, RESULTS / "bank_stream.metadata.json")
FINAL_ZIP = WORK_ROOT / "result_jcam_exp10_predictable_comparator.zip"
with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=(Path("exp10_predictable_comparator") / path.relative_to(RESULTS)).as_posix())
print("Output archive:", FINAL_ZIP, "bytes:", FINAL_ZIP.stat().st_size, "run_ok:", RUN_OK)
